# Random Forest Regression — Exercise 2

**Student:** YOUR NAME  |  **CMSID:** YOUR CMSID

**Dataset:** California Housing Prices

**Dataset link:** https://www.kaggle.com/datasets/camnugent/california-housing-prices

This notebook is a complete implementation of the assigned algorithm. Run cells from top to bottom. Replace the student placeholders before submission.

## 1. Imports

`pandas`/`numpy` manage data; plotting libraries visualize relationships and curves; `train_test_split` creates a holdout; `OneHotEncoder` and `ColumnTransformer` encode the categorical ocean-proximity field; `RandomForestRegressor` fits bootstrap trees with random feature selection; `DecisionTreeRegressor` provides a baseline; and `permutation_importance` measures test-set importance by shuffling one feature.

In [ ]:
# Import data, plotting, preprocessing, ensemble, and metric utilities.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42

## 2. Load, clean, and encode

The target is `median_house_value`; the predictors contain eight numerical measurements and categorical `ocean_proximity`. We impute `total_bedrooms` with the training median inside a pipeline. One-hot encoding changes category labels into numeric indicator columns because scikit-learn tree split comparisons require numeric inputs; treating labels as 0/1/2 would create a false order.

In [ ]:
# Upload housing.csv from Kaggle and load it.
from google.colab import files
# uploaded = files.upload()
CSV_PATH = "housing.csv"
df = pd.read_csv(CSV_PATH)
print(df.shape); display(df.head()); display(df.describe(include="all").T)
print("missing values:\n", df.isna().sum().sort_values(ascending=False).head())

# A quick exploratory relationship plot for the strongest expected geographic signal.
sns.scatterplot(data=df.sample(min(3000, len(df)), random_state=RANDOM_STATE),
                x="median_income", y="median_house_value", hue="ocean_proximity", alpha=.35, legend=False)
plt.title("Income versus house value"); plt.show()

## 3. Randomness, OOB estimation, and training procedure

There are two randomness sources: each tree receives a bootstrap sample of rows, and each node considers only `max_features=m` randomly selected features. `n_estimators=B` is the number of trees. A point is excluded from one bootstrap sample with probability `(1-1/n)^n`, which tends to `e^-1≈0.368`; those out-of-bag predictions estimate generalization without a separate validation fit.

For each tree, draw `n` rows with replacement, grow a tree using SSE reduction, restrict each split to `m` features, and average predictions: `f_RF(x)=(1/B) sum_b T_b(x)`. Averaging lowers variance; smaller `m` can lower tree correlation `rho`, though too small a subset increases individual-tree bias.

In [ ]:
# Separate target and predictors, then create one fixed split for every comparison.
X = df.drop(columns="median_house_value")
y = df["median_house_value"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=RANDOM_STATE)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = ["ocean_proximity"]
preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features)
])

def make_forest(n_estimators=200, max_features=1.0, min_samples_leaf=1):
    # The pipeline prevents imputation/encoding leakage and keeps preprocessing identical.
    forest = RandomForestRegressor(n_estimators=n_estimators, max_features=max_features,
        min_samples_leaf=min_samples_leaf, bootstrap=True, oob_score=True,
        random_state=RANDOM_STATE, n_jobs=-1)
    return Pipeline([("preprocess", preprocessor), ("forest", forest)])

model = make_forest(n_estimators=200, max_features=1.0)
model.fit(X_train, y_train)
forest = model.named_steps["forest"]
print("OOB R^2:", forest.oob_score_)

In [ ]:
# Evaluate any fitted pipeline with common metrics.
def metrics(model, X_eval, y_eval):
    pred = model.predict(X_eval)
    return {"RMSE": mean_squared_error(y_eval, pred, squared=False), "MAE": mean_absolute_error(y_eval, pred), "R2": r2_score(y_eval, pred)}
print("forest test:", metrics(model, X_test, y_test))

# Compare number of trees using OOB and fixed-test error.
B_records = []
for B in [1, 10, 50, 100, 300]:
    trial = make_forest(n_estimators=B, max_features=1.0)
    trial.fit(X_train, y_train)
    fitted = trial.named_steps["forest"]
    # With very small B, some rows have no OOB prediction; exclude undefined rows.
    valid_oob = np.isfinite(fitted.oob_prediction_)
    oob_rmse = mean_squared_error(y_train[valid_oob], fitted.oob_prediction_[valid_oob], squared=False) if valid_oob.any() else np.nan
    B_records.append({"B": B, "OOB_RMSE": oob_rmse,
                      "test_RMSE": metrics(trial, X_test, y_test)["RMSE"]})
B_results = pd.DataFrame(B_records); display(B_results)
B_results.plot(x="B", y=["OOB_RMSE", "test_RMSE"], marker="o", figsize=(8,4), title="Error versus number of trees"); plt.show()

In [ ]:
# Compare feature-subset sizes. After one-hot encoding, d is available from the fitted matrix.
encoded_train = model.named_steps["preprocess"].transform(X_train)
d = encoded_train.shape[1]
max_feature_values = [1, max(1, d // 3), d]
feature_records = []
for m in max_feature_values:
    trial = make_forest(n_estimators=200, max_features=m)
    trial.fit(X_train, y_train)
    feature_records.append({"max_features": m, "test_RMSE": metrics(trial, X_test, y_test)["RMSE"], "oob_R2": trial.named_steps["forest"].oob_score_})
display(pd.DataFrame(feature_records))

# Compare flexible/deep base trees with strongly restricted base trees.
deep_forest = make_forest(n_estimators=200, max_features=1.0, min_samples_leaf=1)
restricted_forest = make_forest(n_estimators=200, max_features=1.0, min_samples_leaf=20)
deep_forest.fit(X_train, y_train); restricted_forest.fit(X_train, y_train)
print("deep:", metrics(deep_forest, X_test, y_test))
print("restricted:", metrics(restricted_forest, X_test, y_test))

## 4. Evaluation, feature importance, and reflection

The forest should be compared with one tree on the identical split. Any improvement is attributed to averaging correlated-but-not-identical trees: `Var(average) = rho*sigma² + (1-rho)*sigma²/B` (equivalently, lower `rho` and larger `B` reduce the ensemble variance). Impurity importance sums weighted SSE reductions; permutation importance measures the test-score deterioration when a feature is shuffled, so it can differ under correlated predictors.

In [ ]:
# Fit the single-tree baseline using the same preprocessing and split.
baseline = Pipeline([("preprocess", preprocessor), ("tree", DecisionTreeRegressor(random_state=RANDOM_STATE))])
baseline.fit(X_train, y_train)
print("single tree:", metrics(baseline, X_test, y_test))
print("random forest:", metrics(model, X_test, y_test))

# Report impurity importances using the transformed feature names.
feature_names = model.named_steps["preprocess"].get_feature_names_out()
impurity = pd.Series(forest.feature_importances_, index=feature_names).sort_values(ascending=False)
display(impurity.head(15).to_frame("impurity_importance"))

# Permutation importance directly measures test-set score loss, using negative RMSE as the scorer.
perm = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1, scoring="neg_root_mean_squared_error")
perm_table = pd.DataFrame({"feature": X_test.columns, "mean_importance": perm.importances_mean, "std": perm.importances_std}).sort_values("mean_importance", ascending=False)
display(perm_table)

## 5. Closing reflection

After running the experiments, state the observed effect of `B`, `m`, and base-tree depth using the printed tables. More trees should stabilize OOB/test error because averaging reduces variance; `m` changes the correlation between trees and therefore the variance term; and large `min_samples_leaf` increases base-tree bias while potentially reducing variance. Tie every claim to your measured RMSE/OOB curves.